# Chapter 1: The Transformer

You'll build an encoder-decoder transformer ("Attention Is All You Need") yourself, from raw PyTorch tensor ops — no `nn.Transformer`, no `nn.MultiheadAttention`. Autograd handles the backward pass; you write the forward mechanics.

**Workflow:** each phase is markdown (concept) -> a code cell with a `TODO` for you to fill in -> a check cell that asserts your implementation is correct. Run the check after you implement; if it raises `NotImplementedError` you haven't filled in the stub yet, if it raises `AssertionError` something is wrong.

Phases:
1. Scaled dot-product attention
2. Multi-head attention
3. Positional encoding
4. Position-wise feed-forward
5. Encoder layer
6. Decoder layer
7. Encoder / decoder stacks
8. Causal masking + full Transformer

In [1]:
import math

import torch
import torch.nn.functional as F
from torch import nn

## 1. Scaled dot-product attention

Given queries $Q$, keys $K$, values $V$:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The $\sqrt{d_k}$ scale keeps the softmax from saturating as $d_k$ grows. `mask` marks positions that must **not** be attended to (set those scores to $-\infty$ before the softmax, so they become 0 after it).

In [2]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q, k, v: (..., seq_len, d_k)
    mask: broadcastable to (..., seq_len, seq_len); where mask == 0, that position must not be attended to.
    returns: (..., seq_len, d_k)
    """

    # TODO: implement Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) 
    # if mask is given, fill masked-out scores with float("-inf") before the softmax
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(q.size(-1))

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
        
    return F.softmax(scores, dim=-1) @ v
    


### Why this matters / where it's used

**Plain-English analogy:** picture a library search. Your **query** ($Q$) is what you typed into the search box. Every book has **keywords** attached to it — its **key** ($K$). The search engine compares your query against every book's keywords to get a relevance score: a good match scores high, an irrelevant book scores low. Then, instead of returning only the single best book, you get a *blend* of every book's **content** ($V$), weighted by how relevant each one was — the best match dominates the blend, irrelevant books contribute almost nothing. That's the whole mechanism; attention just does this for tokens in a sequence instead of books in a library.

**Worked example.** Take the 3-token sentence "The cat sat", with made-up 2D embeddings just for illustration (real models use learned, higher-dimensional $Q$/$K$/$V$ projections — here we skip the projection step to keep the arithmetic simple):

| token | vector |
|---|---|
| The | (1, 0) |
| cat | (0, 1) |
| sat | (1, 1) |

Say the query for "sat" is $(1, 1)$. Its raw similarity score against each key is the dot product $Q \cdot K$:

| key | $Q_{sat} \cdot K$ |
|---|---|
| The = (1,0) | $1$ |
| cat = (0,1) | $1$ |
| sat = (1,1) | $2$ |

Run `softmax([1, 1, 2])` and you get roughly `[0.21, 0.21, 0.58]`. So "sat" attends mostly to itself (58%) and pulls in a little context from "The" and "cat" (21% each). The output for "sat" is then $0.21 \cdot V_{The} + 0.21 \cdot V_{cat} + 0.58 \cdot V_{sat}$ — mostly its own value, blended with a bit of everything else. That's exactly what `scaled_dot_product_attention` computes — just with real learned, $d_k$-dimensional vectors instead of these toy 2D ones, and for every token at once (not just "sat").

Try it yourself: `F.softmax(torch.tensor([1., 1., 2.]), dim=-1)`.

**Flow diagram** (one query's worth — every token runs this in parallel):

```
Q (1 token)        K (all tokens)       V (all tokens)
   |                    |                    |
   +------ Q @ K^T -----+                    |
              |                              |
        scores / sqrt(d_k)                   |
              |                              |
          softmax  -> attention weights      |
              |                              |
              +---------- weights @ V -------+
                          |
                       output (1 token, same shape as V's rows)
```

**Why the scale ($\sqrt{d_k}$) matters:** as $d_k$ grows, dot products grow in magnitude (variance scales with $d_k$ for random vectors). Without scaling, softmax inputs get large, the softmax saturates toward one-hot, and gradients through it vanish — the model stops learning which tokens to attend to. Dividing by $\sqrt{d_k}$ keeps the scores in a range where softmax stays well-behaved.

**Where it's reused, unchanged, throughout this notebook:**
- **Encoder self-attention** (phase 5) — every source token looks at every other source token, no mask.
- **Decoder masked self-attention** (phase 6) — same function, but with a causal mask so a token can't look at the future.
- **Cross-attention** (phase 6) — same function again, but $Q$ comes from the decoder while $K$/$V$ come from the encoder's output. This is literally how the decoder "reads" the source sequence.

This one function is the entire mechanism — multi-head attention (next) is just running it several times in parallel on smaller slices, not a different computation.

**Cost to keep in mind:** $QK^T$ is $O(n^2 \cdot d_k)$ in sequence length $n$ — this quadratic blow-up is *the* reason long-context LLMs need tricks like FlashAttention (same math, fused/tiled for memory bandwidth) or sparse/linear attention variants. Out of scope for this chapter, but it's why "just make the context longer" isn't free.

In [3]:
# check
torch.manual_seed(0)
q = torch.randn(2, 3, 4, 8)
k = torch.randn(2, 3, 4, 8)
v = torch.randn(2, 3, 4, 8)

out = scaled_dot_product_attention(q, k, v)
assert out.shape == (2, 3, 4, 8), out.shape

weights = F.softmax(q @ k.transpose(-2, -1) / math.sqrt(8), dim=-1)
assert torch.allclose(weights.sum(-1), torch.ones(2, 3, 4)), "attention weights must sum to 1 over keys"

# force every query to attend only to key index 2 -> output must equal that key's value
mask = torch.zeros(4, 4)
mask[:, 2] = 1
out_masked = scaled_dot_product_attention(q, k, v, mask)
expected = v[..., 2:3, :].expand(-1, -1, 4, -1)
assert torch.allclose(out_masked, expected, atol=1e-5), "masking is wrong"

print("phase 1 ok")

phase 1 ok


## 2. Multi-head attention

Instead of one attention computation over the full `d_model`, project Q/K/V, split into `n_heads` smaller subspaces, run attention independently per head, then concatenate and project back. The linear layers are already wired up in `__init__` below — your job is `_split_heads` and `forward`.

Hint: by the time you call `scaled_dot_product_attention`, q/k/v should be `(batch, n_heads, seq_len, d_k)`. If you're given a 2D `(seq_len, seq_len)` mask, think carefully about which dim to insert before it'll broadcast correctly against that 4D shape.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # TODO: four nn.Linear(d_model, d_model) layers -- self.q_proj, self.k_proj, self.v_proj, self.out_proj --
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        """(batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)"""

        batch, seq_len, _ = x.size()
        x = x.view(batch, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2) 


    def forward(self, q, k, v, mask=None):
        """
        q, k, v: (batch, seq_len, d_model)
        returns: (batch, seq_len, d_model)
        """
        q = self._split_heads(self.dropout(self.q_proj(q)))
        k = self._split_heads(self.dropout(self.k_proj(k)))
        v = self._split_heads(self.dropout(self.v_proj(v)))

        score = scaled_dot_product_attention(q, k, v, mask)

        score = score.transpose(1, 2).reshape(q.size(0), -1, self.n_heads * self.d_k)

        return self.out_proj(score)

### Why multiple heads? What is a "head" actually doing?

**The problem with a single attention computation:** `scaled_dot_product_attention` is one similarity function over one shared $d_{model}$-dimensional space. But "relevance between tokens" isn't one thing — short-range word order, long-range coreference ("it" referring back to "the cat"), subject-verb agreement, copy-the-previous-occurrence patterns — these are different *kinds* of relationships. One shared Q/K/V projection has to compromise and squeeze all of them into a single similarity metric.

**The fix:** instead of one attention computation over the full $d_{model}$ dimensions, split into `n_heads` independent, smaller attention computations, each over its own $d_k = d_{model} / n_{heads}$-dimensional slice. Each head gets its own slice of the learned Q/K/V projections, so each is free to specialize — empirically, some heads end up tracking adjacent words, others track long-distance dependencies, others (the well-known "induction heads") learn copy-the-previous-occurrence patterns. After running attention independently per head, concatenate every head's output back into a $d_{model}$-dimensional vector and pass it through `out_proj`, a learned layer that mixes information across heads.

**Analogy:** a panel of `n_heads` specialists reading the same sentence at once, each with a narrower field of view ($d_k$ dimensions instead of the full $d_{model}$). One tracks grammar, another tracks who's-talking-about-whom, another tracks position/distance. Once each forms their own opinion (their own attention output), a moderator (`out_proj`) combines all opinions into one verdict.

**Shapes, worked through** ($d_{model}=8$, $n_{heads}=2$ so $d_k=4$, batch=1, seq_len=3):

```
input                      (1, 3, 8)   one sequence, 3 tokens, 8-dim embeddings
  -> q/k/v_proj         ->  (1, 3, 8)   linear projections, shape unchanged
  -> _split_heads       ->  (1, 2, 3, 4)  2 heads, each sees only 4 of the 8 dims
  -> attention per head      (independent! head 0 and head 1 get different attention weights)
  -> merge heads back   ->  (1, 3, 8)   concatenate the 2 heads' 4-dim outputs
  -> out_proj           ->  (1, 3, 8)   learned mix across heads
```

The critical point: heads run **independently**. Head 0's attention weights for token 2 can be completely different from head 1's — that's the entire point, each head is free to attend to something different. If `_split_heads` is wrong (e.g. it slices the wrong dimension), you silently get something that isn't really multi-head attention anymore, even though the shapes might still line up — which is exactly why the check below tests *behavior* (masking), not just output shape.

**Why concatenate-then-project instead of, say, averaging the heads?** Concatenation keeps every head's information untouched; `out_proj` is a *learned* combination, so training decides how to weigh and mix each head's contribution — far more expressive than a fixed average.

In [5]:
# check
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=16, n_heads=4)
x = torch.randn(2, 5, 16)

out = mha(x, x, x)
assert out.shape == (2, 5, 16), out.shape

# every query forced to attend only to position 0 -> every output position must be identical
mask = torch.zeros(5, 5)
mask[:, 0] = 1
out_masked = mha(x, x, x, mask)
assert torch.allclose(out_masked[:, 0], out_masked[:, 4], atol=1e-5), "mask broadcasting is wrong"

print("phase 2 ok")

phase 2 ok


## 3. Positional encoding

Attention has no built-in notion of token order, so we inject position information with fixed sinusoidal encodings, added directly to the token embeddings:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}}) \qquad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]

In [11]:
# check
pe_layer = PositionalEncoding(d_model=8, max_len=100)
assert pe_layer.pe.shape == (100, 8), pe_layer.pe.shape

# position 0: sin(0) = 0, cos(0) = 1, for every frequency
assert torch.allclose(pe_layer.pe[0], torch.tensor([0., 1., 0., 1., 0., 1., 0., 1.]), atol=1e-5)
assert pe_layer.pe.abs().max() <= 1.0 + 1e-6

print("phase 3 ok")

phase 3 ok


## 4. Position-wise feed-forward

A small two-layer MLP applied independently to every position: `Linear -> ReLU -> Dropout -> Linear`.

In [12]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        # TODO: self.net = nn.Sequential(Linear(d_model, d_ff), ReLU, Dropout, Linear(d_ff, d_model))
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
# check
ff = PositionwiseFeedForward(d_model=16, d_ff=32)
x = torch.randn(2, 5, 16)
out = ff(x)
assert out.shape == (2, 5, 16), out.shape
assert any(isinstance(layer, nn.ReLU) for layer in ff.net), "missing nonlinearity"

print("phase 4 ok")

phase 4 ok


## 5. Encoder layer

Self-attention (every source token attends to every other source token), then the feed-forward block — each wrapped in a residual connection + layer norm: `x = norm(x + sublayer(x))`.

In [14]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):
        # TODO: x = norm1(x + self_attn(x, x, x, src_mask)); x = norm2(x + ff(x)); return x

        x = self.norm1(x+ self.self_attn(x, x, x, src_mask))
        x = self.norm2(x + self.ff(x))
        return x


In [15]:
# check
layer = EncoderLayer(d_model=16, n_heads=4, d_ff=32)
x = torch.randn(2, 5, 16)
out = layer(x)
assert out.shape == (2, 5, 16), out.shape

# LayerNorm output should be ~zero-mean / ~unit-variance along the feature dim
assert out.mean(-1).abs().max() < 1e-3, "output doesn't look layer-normed"
assert (out.std(-1, unbiased=False) - 1).abs().max() < 0.2, "output doesn't look layer-normed"

print("phase 5 ok")

phase 5 ok


## 6. Decoder layer

Three sub-blocks: masked self-attention over target tokens, cross-attention (target queries attend over the encoder's `memory`), then feed-forward — each wrapped in residual + layer norm.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, memory, tgt_mask=None, src_mask=None):
        x = self.norm1(x + self.self_attn(x, x, x, tgt_mask))
        x = self.norm2(x + self.cross_attn(x, memory, memory, src_mask))
        x = self.norm3(x + self.ff(x))
        return x


### Why cross-attention? How does the encoder's output actually reach the decoder?

**The problem:** the decoder's self-attention only lets a target token look at *other target tokens generated so far*. That alone gives it zero information about the source sentence — it could happily generate grammatically fluent nonsense with no connection to the input at all. Something has to connect "what the decoder is producing" to "what the encoder read."

**The fix — cross-attention is `scaled_dot_product_attention` again, but with mismatched inputs:** $Q$ comes from the decoder's own (partial) representation, while $K$ and $V$ come from the encoder's `memory`. So the decoder asks, for every position it's generating: *"given what I'm currently producing, which source tokens are relevant?"*

**The actual data flow, concretely:**

```
source tokens --> Encoder (n_layers) --> memory   (batch, src_len, d_model)
                                            |
                                            |  (same memory, reused by EVERY decoder layer)
                                            v
target tokens so far --> [DecoderLayer 1: self_attn -> cross_attn(Q=decoder, K/V=memory) -> ff]
                                            |
                                          [DecoderLayer 2: self_attn -> cross_attn(Q=decoder, K/V=memory) -> ff]
                                            |
                                           ... -> out_proj -> logits
```

The encoder runs **once** per forward pass. Every decoder layer's `cross_attn` attends over that *same* `memory` — the encoder isn't re-run per decoder layer, only the decoder's own representation evolves layer to layer. That's why `Decoder.forward` takes `memory` as an argument rather than computing it itself.

**Worked example — translation alignment.** Say the encoder, after reading "The cat sat", produces these (toy, simplified) memory vectors — one per source token:

| source token | memory vector ($K$ = $V$) |
|---|---|
| The | (1, 0, 0) |
| cat | (0, 1, 0) |
| sat | (0, 0, 1) |

The decoder is generating the French translation and is about to produce the word for "cat" ("chat"). Its query at this step (built from its own self-attention over the partial output so far) happens to be $(0, 1, 0)$ — it's "looking for" whatever corresponds to that direction. Dot products against each source key: `The`$\to 0$, `cat`$\to 1$, `sat`$\to 0$. `softmax([0, 1, 0])` $\approx$ `[0.21, 0.58, 0.21]`. The cross-attention output is dominated by `cat`'s value (58%) — the decoder pulls in mostly the source word "cat" while deciding what to generate. That's word alignment, emerging from nothing more than a dot product and a softmax.

**Sanity check you already have:** the test below changes only `memory` (not `x`) and asserts the output changes — that's the direct proof `cross_attn` is wired to the encoder's output rather than silently ignoring it.

### Other applications of "Q from one place, K/V from another"

Cross-attention is a general pattern, not a translation-specific trick: **whatever you're currently producing/refining supplies $Q$; whatever context you want to pull information from supplies $K$/$V$.** Once you see it that way, it shows up everywhere:

- **Text-to-image diffusion models** (Stable Diffusion, DALL-E 2): the U-Net's cross-attention layers take $Q$ from the image being denoised and $K$/$V$ from a text encoder's embedding of the prompt. This is literally the mechanism by which a text prompt steers image generation — at each denoising step, image regions attend to whichever prompt words are relevant to them.
- **Image captioning / visual question answering**: $Q$ from the text decoder generating the caption/answer, $K$/$V$ from a vision encoder's output (e.g. ViT patch embeddings) — the decoder "looks at" relevant image regions to decide the next word.
- **Speech-to-text** (Whisper): $Q$ from the text decoder, $K$/$V$ from an audio encoder's representation of the spectrogram, aligning text tokens to the relevant audio time-segments.
- **Retrieval-augmented generation (RAG)**: $Q$ from the generator, $K$/$V$ from retrieved document embeddings — the model attends over retrieved passages while generating its answer.
- **Perceiver / Perceiver IO**: $Q$ comes from a small fixed-size learned latent array, $K$/$V$ from a very large raw input (e.g. every pixel of an image). Cross-attention here is used to *compress* — it makes the expensive part of attention scale with the latent size, not the (huge) input size.

In every case it's the same function you already implemented — only the source of $Q$ vs. $K$/$V$ changes.

In [17]:
# check
layer = DecoderLayer(d_model=16, n_heads=4, d_ff=32)
x = torch.randn(2, 5, 16)
memory = torch.randn(2, 7, 16)

out = layer(x, memory)
assert out.shape == (2, 5, 16), out.shape
assert out.mean(-1).abs().max() < 1e-3, "output doesn't look layer-normed"

# output must depend on memory -- proves cross-attention is actually wired to it
out2 = layer(x, torch.randn(2, 7, 16))
assert not torch.allclose(out, out2), "output doesn't depend on memory"

print("phase 6 ok")

phase 6 ok


## 7. Encoder / decoder stacks

Embed tokens, add positional encoding, dropout, then run `n_layers` of the layer type above in sequence. The decoder additionally threads `memory` (and both masks) through every layer.

In [25]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        src = self.embed(src)
        src = self.pos_enc(src)
        src = self.dropout(src)
        for layer in self.layers:
            src = layer(src, src_mask)
        return src


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        tgt = self.embed(tgt)
        tgt = self.pos_enc(tgt)
        tgt = self.dropout(tgt)
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, src_mask)
        return tgt

### Why stack `n_layers` of these, instead of just one?

**One layer = one round of "look around, then transform."** A single self-attention + feed-forward block lets every token gather context from the rest of the sequence once, then nonlinearly transform that. That's a fairly shallow function. Stacking layers means layer 2 operates on the *already-contextualized* output of layer 1, not the raw embeddings — so it can build relationships on top of relationships. In practice this tends to play out like a vision CNN going from edges to shapes to objects: earlier layers pick up local/syntactic patterns, deeper layers pick up longer-range or more abstract structure. More layers = more representational capacity.

**Why this is trainable at all:** naively stacking many nonlinear transformations tends to cause vanishing/exploding gradients — by layer 50, the gradient signal from the loss has been multiplied through 50 nonlinearities and is often useless. The residual connections you implemented in every `EncoderLayer`/`DecoderLayer` (`x = norm(x + sublayer(x))`, not `x = norm(sublayer(x))`) are what prevent this: gradients have a direct additive path back through every layer regardless of what the attention/FFN sublayers do. This is the same trick ResNets use for deep CNNs, and it's *why* transformers can be stacked dozens of layers deep and still train.

**How deep in practice:** the original "Attention Is All You Need" paper used 6 encoder + 6 decoder layers. Modern LLMs push this much further — GPT-3 uses 96 decoder-only layers. More layers generally means more capacity (and cost); this notebook's checks use 2-3 layers purely to keep the forward pass fast to run, not because that's architecturally meaningful.

In [26]:
# check
enc = Encoder(vocab_size=50, d_model=16, n_heads=4, d_ff=32, n_layers=3, max_len=100)
src = torch.randint(0, 50, (2, 6))
memory = enc(src)
assert memory.shape == (2, 6, 16), memory.shape

dec = Decoder(vocab_size=50, d_model=16, n_heads=4, d_ff=32, n_layers=3, max_len=100)
tgt = torch.randint(0, 50, (2, 4))
out = dec(tgt, memory)
assert out.shape == (2, 4, 16), out.shape

print("phase 7 ok")

phase 7 ok


## 8. Causal masking + full Transformer

`causal_mask(seq_len)` builds the mask the decoder's self-attention needs so position `i` can only see positions `<= i` (it must never see future target tokens). Then wire encoder + decoder + a final linear projection to target-vocab logits into `Transformer`.

In [ ]:
def causal_mask(seq_len, device=None):
    """Returns a mask (broadcastable to (batch, n_heads, seq_len, seq_len)) where position i may attend to positions <= i."""
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
    return mask


class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        n_heads=8,
        d_ff=2048,
        n_layers=6,
        max_len=5000,
        dropout=0.1,
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout)
        self.out_proj = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # TODO: encode src -> memory; decode tgt using memory (+ masks); project to vocab logits
        self.memory = self.encoder(src, src_mask)
        self.tgt = self.decoder(tgt, self.memory, tgt_mask, src_mask)
        self.logits = self.out_proj(self.tgt)
        return self.logits


In [34]:
# check
torch.manual_seed(0)
batch, src_len, tgt_len, vocab = 2, 7, 5, 100

model = Transformer(src_vocab_size=vocab, tgt_vocab_size=vocab, d_model=32, n_heads=4, d_ff=64, n_layers=2)
model.eval()  # disable dropout for testing
src = torch.randint(0, vocab, (batch, src_len))
tgt = torch.randint(0, vocab, (batch, tgt_len))

mask = causal_mask(tgt_len)
logits = model(src, tgt, tgt_mask=mask)
assert logits.shape == (batch, tgt_len, vocab), logits.shape

# causal property: changing the LAST target token must not change EARLIER positions' logits
tgt2 = tgt.clone()
tgt2[:, -1] = (tgt2[:, -1] + 1) % vocab
logits2 = model(src, tgt2, tgt_mask=mask)
assert torch.allclose(logits[:, :-1], logits2[:, :-1], atol=1e-5), "earlier positions leaked information from the future"

print("phase 8 ok -- full transformer forward pass works and respects causality")

phase 8 ok -- full transformer forward pass works and respects causality
